# Elasticsearch-inspired distributed search engine — Kaggle runner

Starts a local cluster (one OS process per shard copy + coordinator) and runs the three experiments:

1. **Ranking comparison** vs. a single-node index (global DFS / periodic refresh / shard-local statistics)
2. **Performance scaling** for 1, 2, 4, 8 shards
3. **Failure demonstration**: kill a replica mid-workload, then all copies of a shard

Before running, attach two inputs (Add Input):
* your uploaded code dataset (`dse_kaggle.zip`)
* **parthplc/ms-marco-dataset** — set `CORPUS = None` in the config cell to use the synthetic corpus instead

In [ ]:
# Get the code: attach the uploaded code zip as a Kaggle dataset (Add Input).
# Kaggle unpacks it somewhere under /kaggle/input; find the folder holding dse/
# and copy it to the writable working directory.
import glob, os, shutil, sys
found = glob.glob("/kaggle/input/**/dse/coordinator.py", recursive=True)
assert found, "code dataset not attached: Add Input -> your code dataset"
SRC = os.path.dirname(os.path.dirname(found[0]))
REPO = "/kaggle/working/dse-project"
shutil.copytree(SRC, REPO, dirs_exist_ok=True)
%cd {REPO}
sys.path.insert(0, REPO)

# Most packages are preinstalled on Kaggle; install only what is missing.
import importlib.util
missing = [p for p in ("fastapi", "uvicorn", "httpx", "aiohttp", "nltk") if importlib.util.find_spec(p) is None]
if missing:
    !pip install -q {" ".join(missing)}
print("code from", SRC, "| installed:", missing or "nothing needed")

In [ ]:
# ---- configuration -------------------------------------------------------
# Kaggle dataset "parthplc/ms-marco-dataset" (Add Input -> search "ms-marco-dataset").
# Its CSVs have columns answers, query, finalpassage: passages are the documents,
# the query column gives real MS MARCO test queries.
def find_input(name):
    hits = glob.glob(f"/kaggle/input/**/{name}", recursive=True)
    return next((h for h in hits if "ms-marco" in h), hits[0] if hits else None)

CORPUS  = find_input("train.csv")   # set to None to use the synthetic corpus
QUERIES = find_input("valid.csv")
LIMIT   = None             # max documents to load from CORPUS (None = all)
NUM_DOCS = 200_000         # synthetic corpus size when CORPUS is None
OUT, WORK = "results", "/kaggle/temp/work"
print("corpus:", CORPUS, "| queries:", QUERIES)

def corpus_args():
    if CORPUS:
        a = f"--corpus {CORPUS} --corpus-column finalpassage" + (f" --limit {LIMIT}" if LIMIT else "")
        return a + (f" --queries {QUERIES} --query-column query" if QUERIES else "")
    return f"--num-docs {NUM_DOCS}"
ARGS = f"{corpus_args()} --out {OUT} --work {WORK}"
print(ARGS)

## Quick interactive demo

In [ ]:
from dse import data
from dse.cluster import Cluster

docs = data.load(CORPUS, 50_000, "finalpassage") if CORPUS else data.synthetic_corpus(20_000)
c = Cluster(f"{WORK}/demo", num_shards=4, replicas=1)
print("docs per shard:", c.build(docs))
c.start()
print("allocation:", c.allocation)
print(c.health()["status"])
r = c.search(docs[0][1][:80])
print({k: r[k] for k in ("status", "partial", "shards", "served_by", "took_ms")})
r["hits"][:5]

In [ ]:
# Kill one node: every shard keeps a copy elsewhere, so results are unchanged.
print("killed shards on node-0:", c.kill_node("node-0"))
r = c.search(docs[0][1][:80])
print(r["status"], r["shards"], r["retries"])
# Kill all copies of shard 1: the response is flagged partial.
c.kill_shard(1)
r = c.search(docs[0][1][:80])
print(r["status"], r["partial"], r["shards"])
c.stop()

## Shard allocation and rebalancing

No two copies of a shard share a node. When a node leaves, its copies are rebuilt on the remaining nodes by copying the index from a live peer (including writes made after startup), and a replica is promoted if the primary was lost. When a node joins, copies move onto it until loads differ by at most one. Queries keep running throughout.

In [ ]:
# Rebalancing demo: a node leaves, its shard copies are rebuilt on the other
# nodes from live peers; then a new node joins and copies move onto it.
# A background client keeps querying throughout to show there is no downtime.
import threading, time
import httpx
from collections import Counter

def show(title):
    loads = Counter(n for copies in c.allocation.values() for n in copies)
    print(f"\n== {title}  (cluster {c.health()['status']})")
    for s, copies in sorted(c.allocation.items()):
        print(f"  shard {s}: primary={copies[0]:7s} replicas={copies[1:]}")
    print("  copies per node:", dict(sorted(loads.items())))

def show_plan(plan):
    print("  promoted to primary:", plan.promoted or "-")
    print("  copies removed     :", plan.removed or "-")
    print("  copies rebuilt     :", plan.added or "-")

c = Cluster(f"{WORK}/rebalance", num_shards=4, replicas=1)
c.build(docs)
c.start()
show("initial allocation")

# Writes made after the index files were built: peer recovery must carry them over.
c.index_documents([("live-1", "zzqmarker rebalancing demo document"),
                   ("live-2", "another zzqmarker written after startup")])

# Keep querying in the background while the cluster changes shape.
stats, stop = Counter(), threading.Event()
def client():
    with httpx.Client(timeout=10) as h:
        i = 0
        while not stop.is_set():
            r = h.post(c.url + "/search", json={"query": docs[i % len(docs)][1][:60]})
            stats[r.json().get("status", "error") if r.status_code == 200 else f"http_{r.status_code}"] += 1
            i += 1
t = threading.Thread(target=client, daemon=True)
t.start()
time.sleep(1)

# 1) node-1 leaves: its copies are re-replicated onto the remaining nodes.
t0 = time.time()
plan = c.remove_node("node-1")
show(f"after node-1 left (re-replicated in {time.time() - t0:.1f}s)")
show_plan(plan)
c.wait_green()

# 2) node-4 joins: copies move from the most loaded nodes onto it.
t0 = time.time()
plan = c.add_node("node-4")
show(f"after node-4 joined (rebalanced in {time.time() - t0:.1f}s)")
show_plan(plan)
c.wait_green()

time.sleep(1)
stop.set(); t.join()
print("\nqueries served during the changes:", dict(stats))
hits = {h["doc_id"] for h in c.search("zzqmarker")["hits"]}
print("docs written before the moves still found:", sorted(hits & {"live-1", "live-2"}))
c.stop()

## 1. Ranking comparison

In [ ]:
!python -m experiments.ranking {ARGS} --shards 4 8
import pandas as pd
pd.read_csv(f"{OUT}/ranking_table.csv")

## 2. Performance scaling (1, 2, 4, 8 shards)

In [ ]:
!python -m experiments.scaling {ARGS} --shards 1 2 4 8 --duration 20 --concurrency 16 --delay-ms 1 --jitter-ms 2 --coordinators 2
from IPython.display import Image, display
display(pd.read_csv(f"{OUT}/scaling_summary.csv"))
Image(f"{OUT}/scaling.png")

## 3. Failure demonstration

In [ ]:
!python -m experiments.failover {ARGS} --shards 4 --duration 20
display(pd.read_csv(f"{OUT}/failover_summary.csv"))
Image(f"{OUT}/failover.png")